# 08 – Decision threshold and cost model

The random forest outputs a failure probability; an alarm is raised when it reaches the decision threshold. The threshold is chosen on out-of-fold predictions for the training set (each job is predicted by a model that did not see it, using the same five folds as notebook 07), so the test set is not used.

**Cost model** (assumptions, fixed here before the test set is used):

- every alarm triggers a planned intervention (inspection or tool change): CAD 1,000; a correct alarm prevents the failure
- every missed failure causes an unplanned breakdown: CAD 10,000
- total cost = missed failures × 10,000 + alarms × 1,000

The dataset contains no cost information, so the 10:1 ratio is an assumption; a sensitivity analysis over ratios from 2 to 50 shows how much the conclusions depend on it. The threshold minimising total cost is selected; if several thresholds give the same cost, the highest one (fewest interventions) is used.

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from config import (COST_FAILURE, COST_INTERVENTION, CURRENCY, CV_FOLDS, FAILURE_MODES, FIG_DIR,
                    H5_TOOL_WEAR_RULE, RESULTS_DIR, SEED, SENSITIVITY_RATIOS)
from data import failure_stratum, load_train_test
from features import build_features
from models import make_pipeline

train, _ = load_train_test()
X_train, y_train = build_features(train)

with open(RESULTS_DIR / "07_selected_model.json") as f:
    selection = json.load(f)
selection

## 1. Out-of-fold predictions

In [ ]:
folds = list(StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED).split(X_train, failure_stratum(train)))
pipeline = make_pipeline(selection["model"], selection["features"], **selection["params"])
probability = cross_val_predict(pipeline, X_train, y_train, cv=folds, method="predict_proba")[:, 1]

pd.DataFrame({"UDI": train["UDI"], "failure": y_train, "probability": probability}).to_csv(
    RESULTS_DIR / "08_oof_predictions.csv", index=False)
print(f"Pooled out-of-fold PR-AUC: {average_precision_score(y_train, probability):.3f}")

The pooled PR-AUC differs slightly from the mean of the per-fold scores in notebook 07 because it is computed over all predictions at once.

## 2. Cost for every threshold

In [ ]:
def evaluate(y_true, alarm, cost_failure=COST_FAILURE, cost_intervention=COST_INTERVENTION):
    y_true, alarm = np.asarray(y_true), np.asarray(alarm)
    caught = int((alarm & (y_true == 1)).sum())
    missed = int((~alarm & (y_true == 1)).sum())
    alarms = int(alarm.sum())
    return {
        "alarms": alarms,
        "caught": caught,
        "missed": missed,
        "false alarms": alarms - caught,
        "recall": caught / (caught + missed),
        "precision": caught / alarms if alarms else np.nan,
        "cost": missed * cost_failure + alarms * cost_intervention,
    }


def cost_curve(cost_failure):
    thresholds = np.round(np.arange(0.01, 1.0, 0.01), 2)
    return pd.DataFrame([{"threshold": t, **evaluate(y_train, probability >= t, cost_failure)} for t in thresholds])


def best_threshold(curve):
    return curve.loc[curve["cost"] == curve["cost"].min(), "threshold"].max()


curve = cost_curve(COST_FAILURE)
threshold = best_threshold(curve)
curve.set_index("threshold").loc[[0.05, 0.1, 0.2, threshold, 0.3, 0.5, 0.7]].round(3)

In [ ]:
print(f"Selected threshold: {threshold}")
pd.Series(evaluate(y_train, probability >= threshold)).round(3)

## 3. Comparison with simple maintenance policies (training set, preview)

The same cost model is applied to two simple policies. H5 itself is evaluated on the test set in notebook 09.

In [ ]:
policies = pd.DataFrame({
    "No predictive maintenance": evaluate(y_train, np.zeros(len(y_train), dtype=bool)),
    f"Replace tool at {H5_TOOL_WEAR_RULE} min wear": evaluate(y_train, (X_train["tool_wear"] >= H5_TOOL_WEAR_RULE).to_numpy()),
    "Random forest": evaluate(y_train, probability >= threshold),
}).T
count_columns = ["alarms", "caught", "missed", "false alarms", "cost"]
policies[count_columns] = policies[count_columns].astype(int)
policies.round(3)

## 4. Recall per failure mode (training set, preview)

A job counts towards every mode it has, so jobs with several modes appear in more than one row. H4 is evaluated on the test set in notebook 09.

In [ ]:
alarm = probability >= threshold
pd.DataFrame({
    mode: {"failures": int(train[mode].sum()), "caught": int((alarm & (train[mode] == 1)).sum())}
    for mode in FAILURE_MODES if train[mode].sum() > 0
}).T.assign(recall=lambda d: (d["caught"] / d["failures"]).round(3))

## 5. Sensitivity to the cost ratio

In [ ]:
sensitivity = []
for ratio in SENSITIVITY_RATIOS:
    cost_failure = ratio * COST_INTERVENTION
    t = best_threshold(cost_curve(cost_failure))
    model = evaluate(y_train, probability >= t, cost_failure)
    nothing = evaluate(y_train, np.zeros(len(y_train), dtype=bool), cost_failure)
    rule = evaluate(y_train, (X_train["tool_wear"] >= H5_TOOL_WEAR_RULE).to_numpy(), cost_failure)
    sensitivity.append({
        "cost ratio": ratio,
        "threshold": t,
        "recall": model["recall"],
        "precision": model["precision"],
        "model cost": model["cost"],
        "saving vs no PdM": 1 - model["cost"] / nothing["cost"],
        "saving vs tool rule": 1 - model["cost"] / rule["cost"],
    })
sensitivity = pd.DataFrame(sensitivity).set_index("cost ratio")
sensitivity.to_csv(RESULTS_DIR / "08_cost_sensitivity.csv")
sensitivity.round(3)

## 6. Figures

In [ ]:
INK, MUTED, GRID, GRAY, BLUE, ORANGE = "#0b0b0b", "#52514e", "#e4e3df", "#b9b8b3", "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight", "font.size": 10,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False,
})
chosen = evaluate(y_train, probability >= threshold)

fig, ax = plt.subplots(figsize=(6, 4.2))
precision, recall, _ = precision_recall_curve(y_train, probability)
ax.plot(recall, precision, color=BLUE, linewidth=2, label="Random forest (out-of-fold)")
ax.axhline(y_train.mean(), color=GRAY, linewidth=1.5, linestyle="--", label=f"Random model ({y_train.mean():.3f})")
ax.scatter(chosen["recall"], chosen["precision"], s=70, color=ORANGE, edgecolor="white", linewidth=1, zorder=3,
           label=f"Chosen threshold {threshold}")
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Recall (share of failures caught)")
ax.set_ylabel("Precision (share of alarms that are real)")
ax.set_title("Precision-recall curve (training set, out-of-fold)")
ax.legend(loc="lower left")
fig.tight_layout()
fig.savefig(FIG_DIR / "08_precision_recall.png")

In [ ]:
no_pdm_cost = evaluate(y_train, np.zeros(len(y_train), dtype=bool))["cost"]
rule_cost = policies.loc[f"Replace tool at {H5_TOOL_WEAR_RULE} min wear", "cost"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve["threshold"], curve["cost"] / 1000, color=BLUE, linewidth=2, label="Random forest")
ax.axhline(no_pdm_cost / 1000, color=GRAY, linewidth=1.5, linestyle="--", label="No predictive maintenance")
ax.axhline(rule_cost / 1000, color=INK, linewidth=1, linestyle=":", label=f"Replace tool at {H5_TOOL_WEAR_RULE} min")
ax.scatter(threshold, chosen["cost"] / 1000, s=70, color=ORANGE, edgecolor="white", linewidth=1, zorder=3,
           label=f"Minimum at threshold {threshold}")
ax.set_ylim(0, max(no_pdm_cost, rule_cost) / 1000 * 1.1)
ax.set_xlabel("Decision threshold (alarm if failure probability ≥ threshold)")
ax.set_ylabel(f"Total cost [thousand {CURRENCY}]")
ax.set_title(f"Total cost on the training set (breakdown {COST_FAILURE:,} vs intervention {COST_INTERVENTION:,} {CURRENCY})")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "08_cost_vs_threshold.png")

## 7. Save the decision rule

In [ ]:
decision = {
    "threshold": float(threshold),
    "currency": CURRENCY,
    "cost_intervention": COST_INTERVENTION,
    "cost_failure": COST_FAILURE,
    "oof_recall": chosen["recall"],
    "oof_precision": chosen["precision"],
}
with open(RESULTS_DIR / "08_threshold.json", "w") as f:
    json.dump(decision, f, indent=2)
decision